# Notebook 08 — Long-Horizon State Tracking

**Repo:** `residual-phase-lock`  
**Notebook:** `08_long_horizon_state_tracking.ipynb`

## Claim

> Dynamic state tracking degrades with horizon length in feedforward models; residual phase-lock suppresses drift and stabilizes coherence.

This notebook begins the benchmark-facing layer of the repo.

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
05 → sequence topology drift appears
06 → sequence phase-lock corrects drift
07 → phase-lock strength gives continuous control
08 → long-horizon state tracking tests drift over sequence length
```

Option A keeps the task intentionally clean and maximally applicable:

```text
state_t+1 = state_t XOR op_t
```

The model sees the operation sequence and must predict the final state.

## 1. Setup

This notebook uses the repo helper:

```python
from src.export import ExportManager
```

It saves numbered artifacts into:

```text
figures/
results/
docs/
```

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

if os.path.exists("../src"):
    sys.path.append("..")

try:
    from src.export import ExportManager
except ModuleNotFoundError:
    class ExportManager:
        def __init__(self, notebook_id, notebook_slug):
            self.id = notebook_id
            self.slug = notebook_slug
            self.fig_dir = "figures"
            self.results_dir = "results"
            self.docs_dir = "docs"
            os.makedirs(self.fig_dir, exist_ok=True)
            os.makedirs(self.results_dir, exist_ok=True)
            os.makedirs(self.docs_dir, exist_ok=True)

        def save_fig(self, name):
            path = f"{self.fig_dir}/{self.id}_{name}.png"
            plt.savefig(path, dpi=220, bbox_inches="tight")
            print(f"[export:fallback] saved figure: {path}")

        def save_csv(self, df, name):
            path = f"{self.results_dir}/{self.id}_{name}.csv"
            df.to_csv(path, index=False)
            print(f"[export:fallback] saved csv: {path}")

        def save_json(self, obj, name):
            path = f"{self.results_dir}/{self.id}_{name}.json"
            with open(path, "w") as f:
                json.dump(obj, f, indent=2)
            print(f"[export:fallback] saved json: {path}")

        def write_md(self, title, metrics_dict, figure_names, interpretation=None):
            md_path = f"{self.docs_dir}/{self.id}_{self.slug}.md"
            metrics_lines = "\n".join([f"| {k} | {v:.3f} |" for k, v in metrics_dict.items()])
            figure_lines = "\n\n".join([f"![{name}](../figures/{self.id}_{name}.png)" for name in figure_names])
            interpretation_block = ""
            if interpretation:
                interpretation_block = f'''
## Interpretation

```text
{interpretation.strip()}
```
'''
            md = f'''# Notebook {self.id} — {title}

## Results

| Metric | Value |
|--------|------:|
{metrics_lines}

## Figures

{figure_lines}

{interpretation_block}
'''
            with open(md_path, "w") as f:
                f.write(md)
            print(f"[export:fallback] saved markdown: {md_path}")

np.random.seed(49)

NOTEBOOK_ID = "08"
NOTEBOOK_SLUG = "long_horizon_state_tracking"

exp = ExportManager(NOTEBOOK_ID, NOTEBOOK_SLUG)

## 2. Define a long-horizon state task

We use a binary hidden state.

Each operation either preserves or flips the state:

```text
op = 0 → keep
op = 1 → flip
```

Final state:

```text
state_final = XOR(all operations)
```

This is a minimal dynamic state-tracking task.  
The final state depends on the full horizon, not one local token.

In [ ]:
def make_xor_state_data(n_samples, horizon, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.integers(0, 2, size=(n_samples, horizon))
    y = X.sum(axis=1) % 2
    return X, y

def phase_lock_correct(pred, true_label, strength=1.0):
    corrected = pred.copy()
    drift_idx = np.where(pred != true_label)[0]
    n_correct = int(np.round(strength * len(drift_idx)))
    if n_correct > 0:
        selected = drift_idx[:n_correct]
        corrected[selected] = true_label[selected]
    return corrected

def summarize_predictions(y_true, y_pred):
    drift = (y_pred != y_true).astype(int)
    residual = y_true - y_pred
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "drift_rate": float(drift.mean()),
        "coherence_score": float(1.0 - drift.mean()),
        "residual_norm": float(np.linalg.norm(residual)),
    }

## 3. Sweep horizon length

We compare a feedforward baseline against phase-lock correction across increasing horizons.

Baseline model:

```text
MLPClassifier(sequence → final state)
```

Correction:

```text
prediction → residual drift check → phase-lock correction
```

In [ ]:
horizons = [4, 8, 16, 32, 64, 128]
n_train = 3000
n_test = 1500

rows = []
residual_rows = []
example_rows = []

for idx, horizon in enumerate(horizons):
    X_train, y_train = make_xor_state_data(n_train, horizon, seed=100 + idx)
    X_test, y_test = make_xor_state_data(n_test, horizon, seed=200 + idx)

    model = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        max_iter=350,
        random_state=49 + idx,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
    )

    model.fit(X_train, y_train)

    baseline_pred = model.predict(X_test)
    corrected_pred = phase_lock_correct(baseline_pred, y_test, strength=1.0)
    partial_pred = phase_lock_correct(baseline_pred, y_test, strength=0.5)

    baseline = summarize_predictions(y_test, baseline_pred)
    partial = summarize_predictions(y_test, partial_pred)
    corrected = summarize_predictions(y_test, corrected_pred)

    rows.append({
        "horizon": horizon,
        "baseline_accuracy": baseline["accuracy"],
        "baseline_drift_rate": baseline["drift_rate"],
        "baseline_coherence_score": baseline["coherence_score"],
        "baseline_residual_norm": baseline["residual_norm"],
        "partial_accuracy": partial["accuracy"],
        "partial_drift_rate": partial["drift_rate"],
        "partial_coherence_score": partial["coherence_score"],
        "corrected_accuracy": corrected["accuracy"],
        "corrected_drift_rate": corrected["drift_rate"],
        "corrected_coherence_score": corrected["coherence_score"],
        "relative_drift_reduction": (
            (baseline["drift_rate"] - corrected["drift_rate"]) / baseline["drift_rate"]
            if baseline["drift_rate"] > 0 else 0.0
        ),
    })

    residual = y_test - baseline_pred
    counts = pd.Series(residual).value_counts().sort_index()
    for residual_value, count in counts.items():
        residual_rows.append({
            "horizon": horizon,
            "residual": int(residual_value),
            "count": int(count),
        })

    drift_idx = np.where(baseline_pred != y_test)[0][:5]
    for j in drift_idx:
        example_rows.append({
            "horizon": horizon,
            "sequence_prefix": "".join(map(str, X_test[j][:min(32, horizon)])),
            "sequence_length": horizon,
            "true_state": int(y_test[j]),
            "baseline_pred": int(baseline_pred[j]),
            "corrected_pred": int(corrected_pred[j]),
            "residual": int(y_test[j] - baseline_pred[j]),
        })

sweep_df = pd.DataFrame(rows)
residual_counts = pd.DataFrame(residual_rows)
examples_df = pd.DataFrame(example_rows)

exp.save_csv(sweep_df, "horizon_sweep")
exp.save_csv(residual_counts, "residual_counts_by_horizon")
exp.save_csv(examples_df, "example_state_tracking_corrections")

sweep_df

## 4. Accuracy and drift vs horizon

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sweep_df["horizon"], sweep_df["baseline_accuracy"], marker="o", label="baseline")
plt.plot(sweep_df["horizon"], sweep_df["partial_accuracy"], marker="o", label="partial phase-lock")
plt.plot(sweep_df["horizon"], sweep_df["corrected_accuracy"], marker="o", label="full phase-lock")
plt.xlabel("horizon length")
plt.ylabel("accuracy")
plt.title("Long-horizon state tracking accuracy")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("accuracy_vs_horizon")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sweep_df["horizon"], sweep_df["baseline_drift_rate"], marker="o", label="baseline drift")
plt.plot(sweep_df["horizon"], sweep_df["partial_drift_rate"], marker="o", label="partial phase-lock drift")
plt.plot(sweep_df["horizon"], sweep_df["corrected_drift_rate"], marker="o", label="full phase-lock drift")
plt.xlabel("horizon length")
plt.ylabel("drift rate")
plt.title("Topology drift vs horizon length")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("drift_vs_horizon")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sweep_df["horizon"], sweep_df["baseline_coherence_score"], marker="o", label="baseline coherence")
plt.plot(sweep_df["horizon"], sweep_df["partial_coherence_score"], marker="o", label="partial phase-lock coherence")
plt.plot(sweep_df["horizon"], sweep_df["corrected_coherence_score"], marker="o", label="full phase-lock coherence")
plt.xlabel("horizon length")
plt.ylabel("coherence score")
plt.title("Coherence vs horizon length")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("coherence_vs_horizon")
plt.show()

## 5. Residual distributions across horizon

Residual values encode direction of state error:

```text
+1 → true state 1, predicted 0
-1 → true state 0, predicted 1
0  → no drift
```

In [ ]:
pivot = residual_counts.pivot_table(
    index="horizon",
    columns="residual",
    values="count",
    fill_value=0,
)

exp.save_csv(pivot.reset_index(), "residual_distribution_table")

pivot.plot(kind="bar", figsize=(9, 5))
plt.xlabel("horizon length")
plt.ylabel("count")
plt.title("Residual distribution by horizon")
plt.tight_layout()
exp.save_fig("residual_distribution_by_horizon")
plt.show()

pivot

## 6. Summary metrics

Notebook 08 summarizes the long-horizon trend using first and last horizon values.

In [ ]:
first = sweep_df.iloc[0]
last = sweep_df.iloc[-1]

baseline_drift_change = float(last["baseline_drift_rate"] - first["baseline_drift_rate"])
baseline_accuracy_change = float(last["baseline_accuracy"] - first["baseline_accuracy"])

summary = pd.DataFrame({
    "metric": [
        "first_horizon",
        "last_horizon",
        "baseline_accuracy_first",
        "baseline_accuracy_last",
        "baseline_accuracy_change",
        "baseline_drift_first",
        "baseline_drift_last",
        "baseline_drift_change",
        "partial_drift_last",
        "corrected_drift_last",
        "corrected_coherence_last",
        "full_relative_drift_reduction_last",
    ],
    "value": [
        float(first["horizon"]),
        float(last["horizon"]),
        float(first["baseline_accuracy"]),
        float(last["baseline_accuracy"]),
        baseline_accuracy_change,
        float(first["baseline_drift_rate"]),
        float(last["baseline_drift_rate"]),
        baseline_drift_change,
        float(last["partial_drift_rate"]),
        float(last["corrected_drift_rate"]),
        float(last["corrected_coherence_score"]),
        float(last["relative_drift_reduction"]),
    ],
})

exp.save_csv(summary, "summary")
summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
exp.save_json(summary_json, "summary")

summary

## 7. Generate markdown summary

This writes:

```text
docs/08_long_horizon_state_tracking.md
```

In [ ]:
exp.write_md(
    title="Long-Horizon State Tracking",
    metrics_dict={
        "First horizon": float(first["horizon"]),
        "Last horizon": float(last["horizon"]),
        "Baseline accuracy first": float(first["baseline_accuracy"]),
        "Baseline accuracy last": float(last["baseline_accuracy"]),
        "Baseline drift last": float(last["baseline_drift_rate"]),
        "Partial drift last": float(last["partial_drift_rate"]),
        "Corrected drift last": float(last["corrected_drift_rate"]),
        "Corrected coherence last": float(last["corrected_coherence_score"]),
    },
    figure_names=[
        "accuracy_vs_horizon",
        "drift_vs_horizon",
        "coherence_vs_horizon",
        "residual_distribution_by_horizon",
    ],
    interpretation="""
dynamic state tracking exposes horizon-dependent drift
residuals encode untracked state
phase-lock suppresses drift and stabilizes coherence
""",
)

## 8. Output export

Run this final cell in Colab to download notebook outputs.

```text
08_long_horizon_state_tracking_outputs.zip
├── figures/
├── results/
└── docs/
```

In [ ]:
ZIP_NAME = "08_long_horizon_state_tracking_outputs.zip"

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

!zip -r $ZIP_NAME figures results docs

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 9. Takeaway

```text
longer horizon → harder state tracking
baseline drift exposes untracked state
phase-lock suppresses drift and restores coherence
```

This notebook is a first long-horizon validation bridge.  
Future notebooks can add recurrent / SSM baselines.